# IoT Device Classification Using Machine Learning

**Dataset:** UNSW HomeNet IoT Dataset  
**Platform:** Google Colab

---

## 1. Introduction

### 1.1 Background

The **Internet of Things (IoT)** refers to networks of connected devices embedded with sensors that collect and exchange data over the internet. Examples include smart speakers, security cameras, smart bulbs, and wearable devices.

Despite its advantages, the rapid growth and heterogeneity of IoT devices raises significant **security and network management challenges**:
- Unauthorized devices on networks
- Vulnerability to cyber attacks
- Difficulty in monitoring diverse device types

### 1.2 Problem Statement

**How can we automatically identify and classify IoT devices based on their network traffic patterns?**

Traditional methods rely on manual device registration or MAC address tracking, which are:
- Time-consuming
- Error-prone
- Easily spoofed

**Machine Learning** offers a more robust solution by learning device fingerprints from network traffic features.

### 1.3 Dataset

We use the **UNSW HomeNet IoT Dataset** which contains:
- ~280,000 network flow records
- 14 different IoT device types
- 75+ network traffic features

**Device Types:** Audio, Camera, Hub, Lighting, Motion_Sensor, PC, PowerOutlet, Scale, baby_monitor, power_switch, printer, router, sleep_sensor, smartphone

### 1.4 Objectives

1. Download and explore the UNSW HomeNet IoT dataset
2. Preprocess data for machine learning
3. Train and compare multiple ML models
4. Evaluate model performance
5. Deploy a web demo for real-time classification

---

## 2. Environment Setup

First, let's install the required libraries and import them.

In [ ]:
# Install required packages
!pip install xgboost lightgbm kagglehub -q
print("Packages installed successfully!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("All libraries imported successfully!")

---

## 3. Dataset Download

### 3.1 Kaggle API Setup

**Step 1:** Go to [kaggle.com](https://www.kaggle.com) and sign up

**Step 2:** Click profile -> Settings -> API -> Create New Token

**Step 3:** Upload the downloaded kaggle.json file below

In [ ]:
# Method 1: Upload kaggle.json
from google.colab import files
print("Please upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured!")

In [ ]:
# Method 2: Direct credentials (alternative)
# import os
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

### 3.2 Download Dataset

In [ ]:
import kagglehub

print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("mizanunswcyber/iot-and-non-iot-device-classification-dataset")
print(f"Downloaded to: {dataset_path}")

for f in os.listdir(dataset_path):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / (1024*1024)
    print(f"  - {f} ({size_mb:.1f} MB)")

In [ ]:
csv_file = os.path.join(dataset_path, "UNSW_IoT_Traces.csv")
df = pd.read_csv(csv_file)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

---

## 4. Exploratory Data Analysis

In [ ]:
print("DATASET OVERVIEW")
print(f"Samples: {df.shape[0]:,}")
print(f"Features: {df.shape[1]}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
target_col = 'Type'
print(f"Classes: {df[target_col].nunique()}")
print(df[target_col].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_counts = df[target_col].value_counts()
colors = plt.cm.tab20(np.linspace(0, 1, len(class_counts)))

axes[0].bar(range(len(class_counts)), class_counts.values, color=colors)
axes[0].set_xticks(range(len(class_counts)))
axes[0].set_xticklabels(class_counts.index, rotation=45, ha='right')
axes[0].set_title('Device Type Distribution')

axes[1].pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors)
axes[1].set_title('Device Type Proportions')
plt.tight_layout()
plt.show()

In [ ]:
missing = df.isnull().sum()
if missing.sum() > 0:
    print("Missing values:", missing[missing > 0])
else:
    print("No missing values!")

---

## 5. Data Preprocessing

In [ ]:
initial_rows = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=[target_col])
print(f"Rows: {initial_rows:,} -> {len(df):,}")

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
print("Missing values filled")

In [ ]:
exclude_patterns = ['mac', 'ip', 'address', 'timestamp', 'time', 'date', 'id', 'index', 'name', 'flowid', 'source', 'connection_type', 'unnamed', 'devicename']
cols_to_exclude = [target_col]

for col in df.columns:
    for pattern in exclude_patterns:
        if pattern in col.lower() and col not in cols_to_exclude:
            cols_to_exclude.append(col)
            break

feature_cols = [col for col in df.columns if col not in cols_to_exclude]
feature_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
print(f"Selected {len(feature_cols)} features")

In [ ]:
X = df[feature_cols].values
y = df[target_col].values

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_.tolist()
print(f"Classes: {class_names}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Features scaled")

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)

print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

---

## 6. Model Training

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=20, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=10, learning_rate=0.1, random_state=42, n_jobs=-1, verbose=-1),
    'KNN': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1)
}

In [ ]:
trained_models = {}
training_times = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    start = time.time()
    model.fit(X_train, y_train)
    t = time.time() - start
    trained_models[name] = model
    training_times[name] = t
    val_acc = accuracy_score(y_val, model.predict(X_val))
    print(f"Done in {t:.2f}s | Val Acc: {val_acc:.4f}")

---

## 7. Model Evaluation

In [ ]:
results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'cm': confusion_matrix(y_test, y_pred),
        'y_pred': y_pred
    }
    print(f"{name}: Acc={results[name]['accuracy']:.4f}, F1={results[name]['f1_macro']:.4f}")

In [ ]:
comparison = pd.DataFrame([{
    'Model': name,
    'Accuracy': m['accuracy'],
    'Precision': m['precision'],
    'Recall': m['recall'],
    'F1 (Macro)': m['f1_macro']
} for name, m in results.items()]).sort_values('F1 (Macro)', ascending=False)
print(comparison.to_string(index=False))
print(f"\nBest: {comparison.iloc[0]['Model']}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
width = 0.2
for i, m in enumerate(['Accuracy', 'Precision', 'Recall', 'F1 (Macro)']):
    ax.bar(x + i*width, comparison[m], width, label=m)
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)
ax.set_title('Model Comparison')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for idx, (name, m) in enumerate(results.items()):
    cm = m['cm'].astype('float') / m['cm'].sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[idx], cbar=False)
    axes[idx].set_title(f"{name} (Acc: {m['accuracy']:.4f})")
axes[-1].axis('off')
plt.suptitle('Confusion Matrices', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
xgb_model = trained_models['XGBoost']
imp = pd.DataFrame({'Feature': feature_cols, 'Importance': xgb_model.feature_importances_}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 8))
plt.barh(range(len(imp)), imp['Importance'].values)
plt.yticks(range(len(imp)), imp['Feature'].values)
plt.gca().invert_yaxis()
plt.title('Top 15 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

In [ ]:
best = comparison.iloc[0]['Model']
print(f"Classification Report: {best}")
print(classification_report(y_test, results[best]['y_pred'], target_names=class_names, zero_division=0))

---

## 8. Conclusion

**Key Findings:**
- XGBoost achieved best performance (~88% accuracy)
- Tree-based models outperformed KNN
- Dataset is imbalanced

**Challenges:**
- Motion_Sensor, Lighting: low recall due to class imbalance
- Scale: very few samples

**Future Work:**
- SMOTE for class imbalance
- Deep learning models
- Splunk MLTK integration

---

## 9. Web Demo

**Live Demo:** [https://iot-device-classifier-josie.streamlit.app/](https://iot-device-classifier-josie.streamlit.app/)

**How to Use:**
1. Open the link
2. Upload a PCAP file
3. View classification results
4. Download results as CSV

In [ ]:
from IPython.display import HTML
HTML('<a href="https://iot-device-classifier-josie.streamlit.app/" target="_blank" style="padding:15px 30px;background:#4CAF50;color:white;text-decoration:none;border-radius:5px;font-size:18px">Open Live Demo</a>')

---

## References

1. UNSW HomeNet IoT Dataset: https://www.kaggle.com/datasets/mizanunswcyber/iot-and-non-iot-device-classification-dataset
2. Scikit-learn: https://scikit-learn.org/
3. XGBoost: https://xgboost.readthedocs.io/
4. LightGBM: https://lightgbm.readthedocs.io/